**Dataset:** 118 Indian Unicorn Startups (Kaggle, May 2025)

**Goal:** Clean and normalize the raw CSV into a single production-ready file.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
import json
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
os.getcwd()

'd:\\startup_unicorn_chatbot\\data_preprocessing'

In [3]:
%cd ..

d:\startup_unicorn_chatbot


## 2. Load Raw CSV

In [4]:
RAW_PATH   = 'data/data.csv'
CLEAN_PATH = 'data/unicorns_clean.csv'

os.makedirs('data', exist_ok=True)

df = pd.read_csv(RAW_PATH, encoding="latin-1")
print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Shape  : (118, 19)
Columns: ['Rank', 'Company', 'company_link', 'company_info', 'round_led_by', 'company_background', 'founded_year', 'location', 'country', 'stage', 'primary_sector', 'time_to_unicorn', 'top_investors_1', 'top_investors_2', 'annual_revenue', 'valuation', 'total_funding_till_date', 'employee_count', 'latest_funding_round']


,Rank,Company,company_link,company_info,round_led_by,company_background,founded_year,location,country,stage,primary_sector,time_to_unicorn,top_investors_1,top_investors_2,annual_revenue,valuation,total_funding_till_date,employee_count,latest_funding_round
0,118,Juspay,https://tracxn.com/d/companies/juspay/__fDjQyo...,"Juspay joined the Unicorn Club on Apr 07, 2025...",Kedaara,Provider of a global payments operating system...,2012,Bengaluru,India,Series D,Payments,13 years 3 months,Wellington,Kedaara,"?348Cr as on Mar 31, 2024","?3,450Cr as on Jun 23, 2022",$88M,"1,015 as on Feb 28, 2025","Series D, Apr 07, 2025"
1,117,Money View,https://tracxn.com/d/companies/money-view/__NR...,"Money View joined the Unicorn Club on Sep 12, ...",Accel,Provider of digital financial services includi...,2014,Bengaluru,India,Series E,Alternative Lending,10 years 8 months,Accel,Nexus Venture Partners,"?1,390Cr as on Mar 31, 2024","?10,100Cr as on Sep 17, 2024",$220M,"709 as on Feb 28, 2025","Series E, Sep 12, 2024, $4.65M"
2,116,Ather Energy,https://tracxn.com/d/companies/ather-energy/__...,Ather Energy joined the Unicorn Club on Aug 13...,NIIF,Electric vehicle manufacturer offering scooter...,2013,Bengaluru,India,Public,Electric Vehicles,11 years 7 months,Herald Investment Management,GIC,"?1,790Cr as on Mar 31, 2024",$502M,NaN,"1,630 as on Mar 31, 2025","Series E, Aug 13, 2024, $71.5M"


In [52]:
print(df.isnull().sum()) # check for all null values

Rank                        0
Company                     0
company_link                0
company_info                0
round_led_by               47
company_background          0
founded_year                0
location                    0
country                     0
stage                       0
primary_sector              0
time_to_unicorn             0
top_investors_1             1
top_investors_2             1
annual_revenue              0
valuation                   2
total_funding_till_date    31
employee_count             15
latest_funding_round        0
dtype: int64


## 3. Rename Columns for Consistency

In [5]:
df = df.rename(columns={
    'Company'                  : 'company_name',
    'Rank'                     : 'rank',
    'location'                 : 'city',
    'primary_sector'           : 'sector',
    'top_investors_1'          : 'top_investor_1',
    'top_investors_2'          : 'top_investor_2',
    'annual_revenue'           : 'annual_revenue_raw',
    'valuation'                : 'valuation_raw',
    'total_funding_till_date'  : 'total_funding_raw',
    'employee_count'           : 'employee_count_raw',
    'latest_funding_round'     : 'latest_round',
})
print(f'Renamed columns: {list(df.columns)}')

Renamed columns: ['rank', 'company_name', 'company_link', 'company_info', 'round_led_by', 'company_background', 'founded_year', 'city', 'country', 'stage', 'sector', 'time_to_unicorn', 'top_investor_1', 'top_investor_2', 'annual_revenue_raw', 'valuation_raw', 'total_funding_raw', 'employee_count_raw', 'latest_round']


In [6]:
CITY_MAP = {
    'bengaluru'  : 'bangalore',
    'new delhi'  : 'delhi',
    'gurugram'   : 'gurgaon',
    'noida'      : 'noida',
}

def clean_city(val):  # Trailing whitespace and normalize common variants.
    if pd.isna(val):
        return 'unknown'
    v = str(val).strip().lower()
    return CITY_MAP.get(v, v)

df['city'] = df['city'].apply(clean_city)
print("City distribution:")
print(df['city'].value_counts().to_string())

City distribution:
city
bangalore        46
gurgaon          18
mumbai           15
delhi             5
pune              5
san francisco     4
chennai           4
noida             3
santa clara       2
bellevue          2
singapore         2
salcete           1
hyderabad         1
mountain view     1
palo alto         1
jaipur            1
guwahati          1
new york city     1
ahmedabad         1
thane             1
bethesda          1
san mateo         1
northbrook        1


## 4. Normalize Sector

In [7]:
SECTOR_MAP = {
    'payments'                        : 'fintech',
    'alternative lending'             : 'fintech',
    'insurtech'                       : 'fintech',
    'wealthtech'                      : 'fintech',
    'neobanking'                      : 'fintech',
    'edtech'                          : 'edtech',
    'e-learning'                      : 'edtech',
    'online education'                : 'edtech',
    'healthtech'                      : 'healthtech',
    'telehealth'                      : 'healthtech',
    'health & wellness'               : 'healthtech',
    'e-commerce'                      : 'ecommerce',
    'direct-to-consumer'              : 'ecommerce',
    'd2c'                             : 'ecommerce',
    'logistics'                       : 'logistics',
    'supply chain'                    : 'logistics',
    'mobility'                        : 'mobility',
    'ev'                              : 'mobility',
    'electric vehicles'               : 'mobility',
    'saas'                            : 'saas',
    'enterprise saas'                 : 'saas',
    'b2b saas'                        : 'saas',
    'gaming'                          : 'gaming',
    'media'                           : 'media',
    'social media'                    : 'media',
    'real estate'                     : 'real_estate',
    'proptech'                        : 'real_estate',
    'foodtech'                        : 'foodtech',
    'agritech'                        : 'agritech',
    'spacetech'                       : 'deeptech',
    'deeptech'                        : 'deeptech',
    'aerospace'                       : 'deeptech',
}

def clean_sector(val):
    if pd.isna(val):
        return 'other'
    v = str(val).strip().lower()
    return SECTOR_MAP.get(v, v.replace(' ', '_'))

df['sector'] = df['sector'].apply(clean_sector)
print("Sector distribution after normalization:")
print(df['sector'].value_counts().to_string())

Sector distribution after normalization:
sector
fintech                               16
logistics_tech                         7
horizontal_e-commerce                  6
k-12_edtech                            4
b2b_e-commerce                         4
auto_e-commerce_&_content              4
online_grocery                         4
hrtech                                 3
healthcare_booking_platforms           3
sales_force_automation                 3
investment_tech                        3
internet_first_insurance_platforms     3
banking_tech                           3
e-commerce_enablers                    3
mobility                               2
continued_learning                     2
sports_tech                            2
cryptocurrencies                       2
road_transport_tech                    2
beauty_&_personal_care_products        2
beauty_tech                            2
api_management                         2
business_intelligence                  2
vernacula

## 8. Clean Valuation
Format: `?3,450Cr as on Jun 23, 2022` or `$1.21B`
Extract numeric value — store in **USD billions** for comparability.

In [8]:
def clean_valuation(val):
    """
    Handles:
      '?3,450Cr as on Jun 23, 2022'  -> INR Cr -> USD B (approx /9100)
      '$1.21B'                        -> USD B directly
      '$500M'                         -> USD B
    Returns float USD billions, or None.
    """
    if pd.isna(val) or str(val).strip() in ('', '-', '?'):
        return None
    s = str(val).replace(',', '').strip()

    # USD format: $1.21B or $500M
    usd = re.search(r'\$(\d+\.?\d*)([BM])', s, re.IGNORECASE)
    if usd:
        num, unit = float(usd.group(1)), usd.group(2).upper()
        return round(num if unit == 'B' else num / 1000, 3)

    # INR Cr format: ?3450Cr
    inr = re.search(r'[?₹]?\s*(\d+\.?\d*)\s*Cr', s, re.IGNORECASE)
    if inr:
        cr = float(inr.group(1))
        return round(cr / 9100, 3)   # USD is 91 Rs now

    return None

df['valuation_usd_bn'] = df['valuation_raw'].apply(clean_valuation)
print(f"Parsed: {df['valuation_usd_bn'].notna().sum()} / {len(df)}")
df[['company_name','valuation_raw','valuation_usd_bn']].head(8)

Parsed: 113 / 118


,company_name,valuation_raw,valuation_usd_bn
0,Juspay,"?3,450Cr as on Jun 23, 2022",0.379
1,Money View,"?10,100Cr as on Sep 17, 2024",1.110
2,Ather Energy,$502M,0.502
3,Rapido,"?9,230Cr as on Dec 27, 2024",1.014
4,Porter,"$1B as on May 17, 2024",1.000
5,Perfios,"?8,440Cr as on Mar 27, 2024",0.927
6,Krutrim,"$1B as on Apr 29, 2024",1.000
7,InCred,"$1.04B as on Dec 25, 2023",1.040


In [9]:
def clean_funding(val):
    """
    '$88M'  -> 88.0
    '$1.2B' -> 1200.0
    Returns float USD millions, or None.
    """
    if pd.isna(val) or str(val).strip() in ('', '-'):
        return None
    s = str(val).replace(',', '').strip()
    m = re.search(r'\$(\d+\.?\d*)([BM])', s, re.IGNORECASE)
    if m:
        num, unit = float(m.group(1)), m.group(2).upper()
        return round(num * 1000 if unit == 'B' else num, 1)
    return None

df['total_funding_usd_mn'] = df['total_funding_raw'].apply(clean_funding)
print(f"Parsed: {df['total_funding_usd_mn'].notna().sum()} / {len(df)}")
df[['company_name','total_funding_raw','total_funding_usd_mn']].head(8)

Parsed: 83 / 118


,company_name,total_funding_raw,total_funding_usd_mn
0,Juspay,$88M,88.0
1,Money View,$220M,220.0
2,Ather Energy,NaN,NaN
3,Rapido,$559M,559.0
4,Porter,$151M,151.0
5,Perfios,$263M,263.0
6,Krutrim,$51.1M,51.1
7,InCred,$318M,318.0


In [10]:
def clean_employees_count(val):
    if pd.isna(val):
        return None
    s = str(val).replace(',', '')
    m = re.search(r'(\d+)', s)
    return int(m.group(1)) if m else None

df['employees'] = df['employee_count_raw'].apply(clean_employees_count)
print(f"Parsed: {df['employees'].notna().sum()} / {len(df)}")
df[['company_name','employee_count_raw','employees']].head(8)

Parsed: 103 / 118


,company_name,employee_count_raw,employees
0,Juspay,"1,015 as on Feb 28, 2025",1015.0
1,Money View,"709 as on Feb 28, 2025",709.0
2,Ather Energy,"1,630 as on Mar 31, 2025",1630.0
3,Rapido,"671 as on Feb 28, 2025",671.0
4,Porter,"2,041 as on Feb 28, 2025",2041.0
5,Perfios,"1,072 as on Feb 28, 2025",1072.0
6,Krutrim,"373 as on Jan 31, 2025",373.0
7,InCred,"2,004 as on Jan 31, 2025",2004.0


In [11]:
def clean_revenue(val):
    """
    '?348Cr as on Mar 31, 2024' -> 348.0 (INR Cr)
    '$50M'                      -> convert to INR Cr approx
    Returns float INR Cr, or None.
    """
    if pd.isna(val) or str(val).strip() in ('', '-'):
        return None
    s = str(val).replace(',', '').strip()

    inr = re.search(r'[?₹]?\s*(\d+\.?\d*)\s*Cr', s, re.IGNORECASE)
    if inr:
        return round(float(inr.group(1)), 1)

    usd = re.search(r'\$(\d+\.?\d*)([BM])', s, re.IGNORECASE)
    if usd:
        num, unit = float(usd.group(1)), usd.group(2).upper()
        usd_mn = num * 1000 if unit == 'B' else num
        return round(usd_mn * 9.1, 1)   # approx INR Cr

    return None

df['annual_revenue_cr'] = df['annual_revenue_raw'].apply(clean_revenue)
print(f"Parsed: {df['annual_revenue_cr'].notna().sum()} / {len(df)}")
df[['company_name','annual_revenue_raw','annual_revenue_cr']].head(8)

Parsed: 115 / 118


,company_name,annual_revenue_raw,annual_revenue_cr
0,Juspay,"?348Cr as on Mar 31, 2024",348.0
1,Money View,"?1,390Cr as on Mar 31, 2024",1390.0
2,Ather Energy,"?1,790Cr as on Mar 31, 2024",1790.0
3,Rapido,"?695Cr as on Mar 31, 2024",695.0
4,Porter,"?2,770Cr as on Mar 31, 2024",2770.0
5,Perfios,"?569Cr as on Mar 31, 2024",569.0
6,Krutrim,"?3.09Cr as on Mar 31, 2024",3.1
7,InCred,"?2.71L as on Mar 31, 2024",NaN


In [12]:
def clean_founded_year(val):
    if pd.isna(val):
        return None
    
    m = re.search(r'(\d{4})', str(val))
    return int(m.group(1)) if m else None

df['founded_year'] = df['founded_year'].apply(clean_founded_year)
print(f"Parsed: {df['founded_year'].notna().sum()} / {len(df)}")
print(df['founded_year'].value_counts().sort_index().to_string())

Parsed: 118 / 118
founded_year
1984     1
1996     1
1998     1
2000     4
2004     2
2005     1
2006     1
2007     3
2008     9
2009     3
2010     9
2011    10
2012     8
2013     3
2014    11
2015    18
2016    11
2017     6
2018     7
2019     3
2020     3
2021     2
2023     1


In [13]:
def clean_time_to_unicorn(val):
    """
    '13 years 3 months' -> 159
    '5 years'           -> 60
    Returns total months as int, or None.
    """
    if pd.isna(val):
        return None
    s = str(val).lower()
    years  = re.search(r'(\d+)\s*year',  s)
    months = re.search(r'(\d+)\s*month', s)
    total = 0
    if years:  total += int(years.group(1))  * 12
    if months: total += int(months.group(1))
    return total if total > 0 else None

df['months_to_unicorn'] = df['time_to_unicorn'].apply(clean_time_to_unicorn)
print(f"Parsed: {df['months_to_unicorn'].notna().sum()} / {len(df)}")
df[['company_name','time_to_unicorn','months_to_unicorn']].head(8)

Parsed: 118 / 118


,company_name,time_to_unicorn,months_to_unicorn
0,Juspay,13 years 3 months,159
1,Money View,10 years 8 months,128
2,Ather Energy,11 years 7 months,139
3,Rapido,9 years 6 months,114
4,Porter,10 years 4 months,124
5,Perfios,16 years 2 months,194
6,Krutrim,1 year,12
7,InCred,7 years 10 months,94


In [14]:
def combine_investors(val):
    """combine top_investor_1 and top_investor_2 to a single top_investors"""
    if pd.isna(val) or str(val).strip() in ('', '-', '.', 'N/A'):
        return None
    return str(val).strip()

df['top_investor_1'] = df['top_investor_1'].apply(combine_investors)
df['top_investor_2'] = df['top_investor_2'].apply(combine_investors)

def combine_investors(row):
    investors = [i for i in [row['top_investor_1'], row['top_investor_2']] if i]
    return ', '.join(investors) if investors else 'undisclosed'

df['top_investors'] = df.apply(combine_investors, axis=1)
df[['company_name','top_investor_1','top_investor_2','top_investors']].head(8)

,company_name,top_investor_1,top_investor_2,top_investors
0,Juspay,Wellington,Kedaara,"Wellington, Kedaara"
1,Money View,Accel,Nexus Venture Partners,"Accel, Nexus Venture Partners"
2,Ather Energy,Herald Investment Management,GIC,"Herald Investment Management, GIC"
3,Rapido,WestBridge Capital,Invus,"WestBridge Capital, Invus"
4,Porter,Wellington,Lightstone Ventures,"Wellington, Lightstone Ventures"
5,Perfios,Alternative Lending,Investment Tech,"Alternative Lending, Investment Tech"
6,Krutrim,AI Infrastructure,Z47,"AI Infrastructure, Z47"
7,InCred,KKR,Barclays,"KKR, Barclays"


In [15]:
def unicorn_join_date(val):
    """
    'Juspay joined the Unicorn Club on Apr 07, 2025 after...' -> '2025-04-07'
    Returns ISO date string or None.
    """
    if pd.isna(val):
        return None
    s = str(val)
    
    # Match 'on Mon DD, YYYY' or 'on Mon YYYY'
    m = re.search(
        r'on\s+([A-Za-z]{3,9})\s+(\d{1,2}),?\s+(\d{4})',
        s
    )
    if m:
        try:
            from datetime import datetime
            date_str = f"{m.group(1)} {m.group(2)} {m.group(3)}"
            return datetime.strptime(date_str, '%b %d %Y').strftime('%Y-%m-%d')
        except ValueError:
            pass

    # Fallback: just grab the year
    y = re.search(r'on\s+\w+\s+\d+,?\s+(\d{4})', s)
    return y.group(1) if y else None

df['unicorn_joined_date'] = df['company_info'].apply(unicorn_join_date)
df['unicorn_joined_year'] = df['unicorn_joined_date'].str[:4].apply(
    lambda x: int(x) if pd.notna(x) and x else None
)

print(f"Parsed unicorn join dates: {df['unicorn_joined_date'].notna().sum()} / {len(df)}")
df[['company_name', 'company_info', 'unicorn_joined_date', 'unicorn_joined_year']].head(3)

Parsed unicorn join dates: 118 / 118


,company_name,company_info,unicorn_joined_date,unicorn_joined_year
0,Juspay,"Juspay joined the Unicorn Club on Apr 07, 2025...",2025-04-07,2025
1,Money View,"Money View joined the Unicorn Club on Sep 12, ...",2024-09-12,2024
2,Ather Energy,Ather Energy joined the Unicorn Club on Aug 13...,2024-08-13,2024


In [16]:
def clean_round_led_by(val):
    if pd.isna(val) or str(val).strip() in ('', '-', '.', 'N/A'):
        return None
    return str(val).strip()

df['round_led_by'] = df['round_led_by'].apply(clean_round_led_by)

print(f"round_led_by non-null: {df['round_led_by'].notna().sum()} / {len(df)}")
df[['company_name', 'round_led_by']].dropna().head(10)

round_led_by non-null: 71 / 118


,company_name,round_led_by
0,Juspay,Kedaara
1,Money View,Accel
2,Ather Energy,NIIF
5,Perfios,Ontario Teachers' Pension Plan
6,Krutrim,Z47
7,InCred,Ranjan Pai
8,Zepto,StepStone Group
9,boAt,Warburg Pincus
13,OneCard,Temasek
15,Purplle,Paramark Ventures


In [17]:
def clean_latest_round(val):
    if pd.isna(val) or str(val).strip() in ('', '-'):
        return pd.Series({'round_stage': None, 'round_year': None, 'round_amount_mn': None})

    s = str(val).strip()

    # Stage: first token before comma
    stage_m = re.match(r'^([^,]+)', s)
    stage = stage_m.group(1).strip() if stage_m else None

    # Year
    year_m = re.search(r'(\d{4})', s)
    year = int(year_m.group(1)) if year_m else None

    # Amount
    amt_m = re.search(r'\$(\d+\.?\d*)([BM])', s, re.IGNORECASE)
    amount = None
    if amt_m:
        num, unit = float(amt_m.group(1)), amt_m.group(2).upper()
        amount = round(num * 1000 if unit == 'B' else num, 1)

    return pd.Series({'round_stage': stage, 'round_year': year, 'round_amount_mn': amount})

round_df = df['latest_round'].apply(clean_latest_round)
df = pd.concat([df, round_df], axis=1)
df[['company_name','latest_round','round_stage','round_year','round_amount_mn']].head(8)

,company_name,latest_round,round_stage,round_year,round_amount_mn
0,Juspay,"Series D, Apr 07, 2025",Series D,2025,NaN
1,Money View,"Series E, Sep 12, 2024, $4.65M",Series E,2024,4.7
2,Ather Energy,"Series E, Aug 13, 2024, $71.5M",Series E,2024,71.5
3,Rapido,"Series E, Dec 27, 2024, $29.8M",Series E,2024,29.8
4,Porter,"Series F, May 08, 2025",Series F,2025,NaN
5,Perfios,"Series D, Mar 13, 2024, $80M",Series D,2024,80.0
6,Krutrim,"Angel, Apr 12, 2024, $99.6K",Angel,2024,NaN
7,InCred,"Conventional Debt, Mar 13, 2025, $30M",Conventional Debt,2025,30.0


In [18]:
null_bg = df['company_background'].isna().sum()
print(f"Null company_background: {null_bg}")

if null_bg > 0:
    # fallback: build from available fields
    def build_fallback_bg(row):
        parts = []
        if pd.notna(row.get('company_name')):
            parts.append(f"{row['company_name']} is a startup")
        if pd.notna(row.get('sector')):
            parts.append(f"in the {row['sector']} sector")
        if pd.notna(row.get('city')):
            parts.append(f"based in {row['city']}")
        return ' '.join(parts) or 'Indian unicorn startup'

    mask = df['company_background'].isna()
    df.loc[mask, 'company_background'] = df[mask].apply(build_fallback_bg, axis=1)

# Verify lengths
df['bg_length'] = df['company_background'].str.len()
print('Background length — min:', df['bg_length'].min(), 'max:', df['bg_length'].max(), 'mean:', round(df['bg_length'].mean()))
df.drop(columns=['bg_length'], inplace=True)
print()


Null company_background: 0
Background length — min: 115 max: 914 mean: 416



## 17. Select Final Columns

In [19]:
FINAL_COLS = [
    # ── Identity ──────────────────────────────────────────────
    'rank',
    'company_name',
    'company_link',           # tracxn reference URL
    'country',                # all India — kept for LlamaIndex completeness

    # ── Rich text (ChromaDB embedding + display) ───────────────
    'company_background',     # → ChromaDB document (embedding text)
    'company_info',           # original unicorn join sentence

    # ── Extracted from company_info ────────────────────────────
    'unicorn_joined_date',    # e.g. '2025-04-07'
    'unicorn_joined_year',    # e.g. 2025 (int, filterable)

    # ── Sector / location ──────────────────────────────────────
    'sector',                 # normalized         
    'city',                   # normalized
    'stage',

    # ── Founding ───────────────────────────────────────────────
    'founded_year',
    'time_to_unicorn',        # original string e.g. '13 years 3 months'
    'months_to_unicorn',      # numeric total months

    # ── Investors ──────────────────────────────────────────────
    'round_led_by',           # lead investor for latest round
    'top_investor_1',
    'top_investor_2',
    'top_investors',          # combined clean string

    # ── Financials — numeric (for LlamaIndex) ──────────────────
    'valuation_usd_bn',
    'total_funding_usd_mn',
    'annual_revenue_cr',
    'employees',


    # ── Latest round ───────────────────────────────────────────
    'latest_round',           # original string
    'round_stage',            # extracted e.g. 'Series D'
    'round_year',             # extracted int
    'round_amount_mn',        # extracted float USD mn
]

df_clean = df[[c for c in FINAL_COLS if c in df.columns]].copy()
print(f'Final shape: {df_clean.shape}')
print(f'Columns kept: {len(df_clean.columns)}')
df_clean.dtypes

Final shape: (118, 26)
Columns kept: 26


rank                      int64
company_name             object
company_link             object
country                  object
company_background       object
company_info             object
unicorn_joined_date      object
unicorn_joined_year       int64
sector                   object
city                     object
stage                    object
founded_year              int64
time_to_unicorn          object
months_to_unicorn         int64
round_led_by             object
top_investor_1           object
top_investor_2           object
top_investors            object
valuation_usd_bn        float64
total_funding_usd_mn    float64
annual_revenue_cr       float64
employees               float64
latest_round             object
round_stage              object
round_year                int64
round_amount_mn         float64
dtype: object

In [20]:
def build_embedding_text(row): # This will help to create the text we need for semantic search 
    parts = []

    # ── Identity ─────────────────────────────
    if pd.notna(row.get('company_name')):
        parts.append(f"{row['company_name']} is an Indian unicorn startup.")

    if pd.notna(row.get('sector')) and row['sector'] != 'unknown':
        parts.append(f"It operates in the {row['sector']} sector.")

    if pd.notna(row.get('city')) and row['city'] != 'unknown':
        parts.append(f"It is based in {str(row['city']).title()}.")

    if pd.notna(row.get('stage')):
        parts.append(f"Current stage: {row['stage']}.")

    # ── Background (Primary Embedding Content) ─────────────────
    if pd.notna(row.get('company_background')):
        parts.append(str(row['company_background']).strip())

    if pd.notna(row.get('company_info')):
        parts.append(str(row['company_info']).strip())

    # ── Founding ─────────────────────────────
    if pd.notna(row.get('founded_year')):
        parts.append(f"Founded in {int(row['founded_year'])}.")

    if pd.notna(row.get('time_to_unicorn')):
        parts.append(f"Time taken to reach unicorn status: {row['time_to_unicorn']}.")

    # ── Unicorn Details ──────────────────────
    if pd.notna(row.get('unicorn_joined_date')):
        parts.append(f"Joined the unicorn club on {row['unicorn_joined_date']}.")

    if pd.notna(row.get('unicorn_joined_year')):
        parts.append(f"Became a unicorn in {int(row['unicorn_joined_year'])}.")

    # ── Investors ────────────────────────────
    investors = []

    if pd.notna(row.get('top_investor_1')):
        investors.append(row['top_investor_1'])

    if pd.notna(row.get('top_investor_2')):
        investors.append(row['top_investor_2'])

    if pd.notna(row.get('round_led_by')):
        investors.append(f"latest round led by {row['round_led_by']}")

    if investors:
        parts.append("Investors include " + ", ".join(investors) + ".")

    # ── Financials ───────────────────────────
    if pd.notna(row.get('valuation_usd_bn')):
        parts.append(f"Valuation: {row['valuation_usd_bn']} billion USD.")

    if pd.notna(row.get('total_funding_usd_mn')):
        parts.append(f"Total funding raised: {row['total_funding_usd_mn']} million USD.")

    if pd.notna(row.get('annual_revenue_cr')):
        parts.append(f"Annual revenue: {row['annual_revenue_cr']} crore INR.")

    if pd.notna(row.get('employees')) and row['employees'] > 0:
        parts.append(f"It employs approximately {int(row['employees'])} people.")

    # ── Latest Round ─────────────────────────
    if pd.notna(row.get('round_stage')):
        parts.append(f"Latest funding round: {row['round_stage']}.")

    if pd.notna(row.get('round_year')):
        parts.append(f"The round took place in {int(row['round_year'])}.")

    if pd.notna(row.get('round_amount_mn')):
        parts.append(f"Round amount: {row['round_amount_mn']} million USD.")

    return " ".join(parts)

In [21]:
df_clean['embedding_text'] = df_clean.apply(build_embedding_text, axis=1)

In [22]:
# check lengths of our new text
print(
    "Min length:",
    df_clean['embedding_text'].str.len().min(),
    "Max length:",
    df_clean['embedding_text'].str.len().max(),
    "Mean length:",
    int(df_clean['embedding_text'].str.len().mean())
)

Min length: 645 Max length: 1528 Mean length: 1063


In [23]:
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(df_clean)} rows → {CLEAN_PATH}")
df_clean.head(3)

Saved 118 rows → data/unicorns_clean.csv


,rank,company_name,company_link,country,company_background,company_info,unicorn_joined_date,unicorn_joined_year,sector,city,...,top_investors,valuation_usd_bn,total_funding_usd_mn,annual_revenue_cr,employees,latest_round,round_stage,round_year,round_amount_mn,embedding_text
0,118,Juspay,https://tracxn.com/d/companies/juspay/__fDjQyo...,India,Provider of a global payments operating system...,"Juspay joined the Unicorn Club on Apr 07, 2025...",2025-04-07,2025,fintech,bangalore,...,"Wellington, Kedaara",0.379,88.0,348.0,1015.0,"Series D, Apr 07, 2025",Series D,2025,NaN,Juspay is an Indian unicorn startup. It operat...
1,117,Money View,https://tracxn.com/d/companies/money-view/__NR...,India,Provider of digital financial services includi...,"Money View joined the Unicorn Club on Sep 12, ...",2024-09-12,2024,fintech,bangalore,...,"Accel, Nexus Venture Partners",1.110,220.0,1390.0,709.0,"Series E, Sep 12, 2024, $4.65M",Series E,2024,4.7,Money View is an Indian unicorn startup. It op...
2,116,Ather Energy,https://tracxn.com/d/companies/ather-energy/__...,India,Electric vehicle manufacturer offering scooter...,Ather Energy joined the Unicorn Club on Aug 13...,2024-08-13,2024,mobility,bangalore,...,"Herald Investment Management, GIC",0.502,NaN,1790.0,1630.0,"Series E, Aug 13, 2024, $71.5M",Series E,2024,71.5,Ather Energy is an Indian unicorn startup. It ...
